# Gravitational Waves in Code: Generating CBC Signals with Sage

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nnarenraju/sage/blob/main/notebooks/colab/01_signal_generation.ipynb)

This notebook introduces gravitational-wave (GW) signals from compact binary coalescences (CBCs) and shows how to generate them using Sage's GPU-native `IMRPhenomD` waveform approximant.

**What you will learn:**
- What a CBC waveform looks like in time and frequency
- How chirp mass controls signal duration and bandwidth
- How spin shifts the merger frequency
- How to compute the optimal network SNR with a design detector PSD

**Runtime:** ~8 min on a Colab T4 GPU

## Setup

Install Sage and its dependencies (first run only — takes ~3 minutes).

In [ ]:
# Install Sage from GitHub (run once per Colab session)
import subprocess, sys

try:
    import sage
    print('Sage already installed.')
except ImportError:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/nnarenraju/sage.git'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'sage/'], check=True)
    print('Sage installed.')


In [ ]:
import warnings
warnings.filterwarnings('ignore', 'Wswiglal-redir-stdio')

import numpy as np
import matplotlib.pyplot as plt
import torch

from pycbc.psd import aLIGOZeroDetHighPower
from sage.core.base_classes import BaseConfig, BaseDataConfig
from sage.core.config import register_configs
from sage.data.waveform.approximants.IMRPhenomD import IMRPhenomD
from sage.data.waveform.project import ConstantProjection

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')


In [ ]:
# Sage requires a configuration object to be registered before using
# any module that reads detector or data parameters.

class TutorialCFG:
    batch_size    = 32
    device        = device
    dtype         = torch.float32
    detectors     = ['H1', 'L1']
    do_point_estimate = []
    class_balance = 0.5
    clip_norm     = 1.0
    autocast      = False

class TutorialDataCFG:
    sample_rate               = 2048.0
    signal_low_frequency_cutoff = 20.0
    sample_length_in_s        = 8.0
    padding_length_in_s       = 2.0

register_configs(BaseConfig(TutorialCFG()), BaseDataConfig(TutorialDataCFG()))
print('Configs registered.')


## 1. Frequency Grid and Waveform Generation

Sage's `IMRPhenomD` works in the frequency domain. We build a linearly-spaced grid from `f_low` to `f_high` and pass it (along with a reference frequency `f_ref`) to the waveform generator. The generator returns the plus and cross polarisations `(hp, hc)` for each set of binary parameters.

After calling `pad_missing_frequencies`, the output array runs from 0 Hz to `f_high`, with zeros below `f_low`.

In [ ]:
# Frequency grid: 20 Hz to 1024 Hz, df = 1/16 Hz
f_l, f_u, del_f = 20.0, 1024.0, 1.0 / 16.0
n_waveform = int(round((f_u - f_l) / del_f)) + 1   # bins in [f_l, f_u]
n_padded   = int(round(f_u / del_f)) + 1            # bins in [0, f_u] after padding

print(f'Waveform grid: {n_waveform} bins ({f_l}–{f_u} Hz, df={del_f:.4f} Hz)')
print(f'Padded output: {n_padded} bins (0–{f_u} Hz)')

# Aero-design PSD for SNR calculations (no file dependency)
psd_np  = np.array(aLIGOZeroDetHighPower(n_padded, del_f, f_l).data[:])
psd_np[:int(f_l / del_f)] = np.inf   # below f_l: signal is zero, avoid /0
asd_np  = np.sqrt(psd_np)
psd_dev = torch.tensor(psd_np, dtype=torch.float64, device=device)
asd_dev = torch.tensor(asd_np, dtype=torch.float64, device=device)

freqs_padded = torch.arange(n_padded, dtype=torch.float64, device=device) * del_f
print('PSD ready.')


In [ ]:
# Parameters: [m1, m2, chi1z, chi2z, distance_Mpc, tc, phic, inclination, polarization, ra, dec]
# Three representative binaries:
#   light:  7+7  Msun  (long chirp, low frequency)
#   medium: 30+20 Msun (typical training event)
#   heavy:  50+40 Msun (short, high-frequency merger)

configs = [
    dict(label='Light  (7+7 M☉)',   m1=7.,  m2=7.,  chi1=0., chi2=0.),
    dict(label='Medium (30+20 M☉)', m1=30., m2=20., chi1=0., chi2=0.),
    dict(label='Heavy  (50+40 M☉)', m1=50., m2=40., chi1=0., chi2=0.),
]

dist_mpc, tc, phic = 400., 0., 0.
incl, pol, ra, dec = np.pi / 3., 0.5, 1.2, 0.4

waveforms = {}  # store (hp, hc) per label

for cfg_w in configs:
    params = torch.tensor(
        [[cfg_w['m1'], cfg_w['m2'], cfg_w['chi1'], cfg_w['chi2'],
          dist_mpc, tc, phic, incl, pol, ra, dec]],
        dtype=torch.float64, device=device
    )
    f = (f_l + del_f * torch.arange(n_waveform, dtype=torch.float64, device=device)) \
        .unsqueeze(0).clone()
    f_ref = torch.full((1, 1), f_l, dtype=torch.float64, device=device)

    hp, hc = IMRPhenomD(f, f_ref)(params, reproduce_lal=True)
    # hp, hc: shape (1, n_padded) complex64
    waveforms[cfg_w['label']] = (hp.squeeze(0).cpu().numpy(),
                                  hc.squeeze(0).cpu().numpy())
    print(f"{cfg_w['label']}: generated ({n_padded} freq bins)")


## 2. Time-Domain Waveforms

We inverse-FFT the frequency-domain plus polarisation to get the time-domain strain. Heavier binaries merge faster, producing shorter chirps; lighter binaries sweep through the detector band for many seconds.

In [ ]:
dt = 1.0 / (2.0 * f_u)     # Nyquist dt for f_u
T  = n_padded / (f_u / del_f + 1) / del_f   # approx segment length
n_td = 2 * (n_padded - 1)   # time-domain length after irfft
t_axis = np.arange(n_td) * dt - n_td * dt   # zero at end (merger)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
colors = ['C0', 'C1', 'C2']

for ax, (label, (hp, _)), color in zip(axes, waveforms.items(), colors):
    hp_td = np.fft.irfft(hp)   # (n_td,) real
    # Normalise to unit peak for comparison
    hp_td /= np.max(np.abs(hp_td)) + 1e-30
    ax.plot(t_axis, hp_td, color=color, lw=0.8)
    ax.set_ylabel('h+ (normalised)')
    ax.set_title(label)
    ax.set_xlim(-8, 0.1)
    ax.axvline(0, color='k', lw=0.5, ls='--', label='merger')
    ax.legend(loc='upper left', fontsize=8)

axes[-1].set_xlabel('Time before merger (s)')
plt.tight_layout()
plt.show()


## 3. Frequency-Domain Amplitude Spectra

The amplitude spectral density (ASD) of each waveform shows where the signal power is concentrated. The chirp mass $\mathcal{M}$ determines the merger (ISCO) frequency — heavier binaries merge at lower frequencies.

In [ ]:
freqs = np.arange(n_padded) * del_f

fig, ax = plt.subplots(figsize=(10, 5))

# Detector ASD
mask = (freqs >= f_l) & (freqs <= f_u)
ax.loglog(freqs[mask], asd_np[mask], 'k--', lw=1.5, label='aLIGO design ASD', zorder=5)

for (label, (hp, _)), color in zip(waveforms.items(), colors):
    amp = np.abs(hp)
    ax.loglog(freqs[mask], amp[mask], color=color, lw=1.5, label=label)

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Strain amplitude |h(f)|')
ax.set_xlim(f_l, f_u)
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Spin Effects on Waveform

Aligned spins (chi1z, chi2z) shift the merger frequency. Positive (aligned) spin delays the merger — the binary can orbit closer before plunging — and raises the peak frequency. Negative spin has the opposite effect.

In [ ]:
spin_configs = [
    dict(label='chi1z = -0.99 (anti-aligned)', chi1=-0.99, chi2=-0.99),
    dict(label='chi1z =  0.00 (non-spinning)',  chi1= 0.00, chi2= 0.00),
    dict(label='chi1z = +0.99 (aligned)',       chi1=+0.99, chi2=+0.99),
]

spin_waveforms = {}
for sc in spin_configs:
    params = torch.tensor(
        [[30., 20., sc['chi1'], sc['chi2'], dist_mpc, tc, phic, incl, pol, ra, dec]],
        dtype=torch.float64, device=device
    )
    f = (f_l + del_f * torch.arange(n_waveform, dtype=torch.float64, device=device)).unsqueeze(0).clone()
    f_ref = torch.full((1, 1), f_l, dtype=torch.float64, device=device)
    hp, hc = IMRPhenomD(f, f_ref)(params, reproduce_lal=True)
    spin_waveforms[sc['label']] = hp.squeeze(0).cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 5))
for (label, hp), color in zip(spin_waveforms.items(), ['C3', 'C4', 'C5']):
    hp_td = np.fft.irfft(hp)
    hp_td /= np.max(np.abs(hp_td)) + 1e-30
    ax.plot(t_axis[-1000:], hp_td[-1000:], color=color, lw=1.2, label=label)

ax.set_xlabel('Time (last 0.5 s before merger)')
ax.set_ylabel('h+ (normalised)')
ax.set_title('Spin effect on merger phase (30+20 M☉)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Optimal Network SNR

The optimal SNR tells us how detectable a signal would be if matched filtering were perfect:

$$\rho^2 = \frac{4}{\Delta f} \sum_f \frac{|h(f)|^2}{S_n(f)}$$

We project onto H1 and L1 and sum the per-detector SNRs in quadrature. Here we compute how SNR varies with distance for each of our three mass configurations.

In [ ]:
pwave = ConstantProjection()

distances_Mpc = np.logspace(1, 4, 40)  # 10 Mpc to 10 Gpc

fig, ax = plt.subplots(figsize=(9, 5))

for (label, (hp, hc)), color in zip(waveforms.items(), colors):
    hp_t = torch.tensor(hp, device=device).unsqueeze(0)  # (1, n_padded)
    hc_t = torch.tensor(hc, device=device).unsqueeze(0)
    ra_t  = torch.tensor([ra],  dtype=torch.float64, device=device)
    dec_t = torch.tensor([dec], dtype=torch.float64, device=device)
    pol_t = torch.tensor([pol], dtype=torch.float64, device=device)

    signal = pwave(hp_t, hc_t, ra=ra_t, dec=dec_t, polarization=pol_t)
    # signal: (1, 2, n_padded) at dist_mpc = 400 Mpc

    h_sq = signal.abs().pow(2).sum(dim=1)          # sum over detectors: (1, n_padded)
    rho_sq_at_400 = (4.0 / del_f) * (h_sq / psd_dev).sum().item()
    rho_at_400 = rho_sq_at_400 ** 0.5

    # SNR scales as 1/distance
    snrs = rho_at_400 * dist_mpc / distances_Mpc
    ax.loglog(distances_Mpc, snrs, color=color, lw=2, label=label)

ax.axhline(8, color='k', ls='--', lw=1, label='SNR = 8 threshold')
ax.set_xlabel('Luminosity distance (Mpc)')
ax.set_ylabel('Optimal network SNR')
ax.set_title('Optimal network SNR vs. distance (face-on, aLIGO design)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Summary Table

A compact overview of the key properties of each binary at 400 Mpc.

In [ ]:
from pycbc.conversions import mchirp_from_mass1_mass2

print(f'{"Binary":25s}  {"Mchirp (Msun)":>14}  {"Opt. Net. SNR @ 400 Mpc":>24}')
print('-' * 70)

for cfg_w, (label, (hp, hc)) in zip(configs, waveforms.items()):
    mc = mchirp_from_mass1_mass2(cfg_w['m1'], cfg_w['m2'])

    hp_t = torch.tensor(hp, device=device).unsqueeze(0)
    hc_t = torch.tensor(hc, device=device).unsqueeze(0)
    ra_t  = torch.tensor([ra],  dtype=torch.float64, device=device)
    dec_t = torch.tensor([dec], dtype=torch.float64, device=device)
    pol_t = torch.tensor([pol], dtype=torch.float64, device=device)

    signal = pwave(hp_t, hc_t, ra=ra_t, dec=dec_t, polarization=pol_t)
    h_sq   = signal.abs().pow(2).sum(dim=1)
    rho    = ((4.0 / del_f) * (h_sq / psd_dev).sum()).sqrt().item()

    print(f'{label:25s}  {mc:>14.2f}  {rho:>24.1f}')
